
# Benchmark GCN on LS_a


### Introduction

This notebook walks through reproducing the key results from the paper for the search space `ls_a`.

## 1. Import Dependencies

First, we import the necessary dependencies for our evaluation.

In [1]:
# Suppress PyTorch Warnings and ensure proper dependency loading
import warnings
import os
import sys

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../..')) # Go two levels up from the notebook location to reach project root

if project_root not in sys.path:
    sys.path.insert(0, project_root)
print("Project root set to:", project_root)

Project root set to: C:\SQuASH


In [6]:
import os
import torch
import json
from torch_geometric.data import DataLoader
import numpy as np

from config import DeviceConfig, get_default_model_config_by_search_space, PathConfig, QCConfig, \
    get_model_config_from_path
from evaluate.evaluate_model import run_gcn_evaluation
from util.config_utils import get_gate_set_and_features_by_name
from util.data_loader import load_data
from surrogate_models.architectures.gnn.gcn_runner import prepare_paths_and_config, set_seed
from surrogate_models.architectures.gnn.gnn_model import RegGNN
from evaluate.evaluate_utils import compute_metrics
from evaluate.evaluate_utils import load_gnn_model, evaluate_gnn


In [7]:
# Initialize device configuration and path configuration
device_config = DeviceConfig()
device        = device_config.device
path_config   = PathConfig()

## 2. Data Loading Utils

Similar to the previous notebook, we define function to load corresponding test data from [Zenodo](https://zenodo.org/records/15551805).  
> ⚠️ **Note: If you've already downloaded the data before, please skip this section and data loading cells in the evaluation.**

In [ ]:
!pip install requests

In [3]:
import requests
import zipfile
import io
import os

In [8]:
def zenodo_load_data():
    dataset_names = [
    f"test_{search_space}_squash"
    ]
    for d in dataset_names:
        target_file_name = f'{d}.pt'
        target_path = os.path.join(target_directory, target_file_name)
        if os.path.isfile(target_path):
            print(f"File '{target_file_name}' already exists at '{target_directory}'. Skipping download.")
        else:
            zip_url = f"https://zenodo.org/record/15551805/files/graph_data_{search_space}.zip"
        
            print(f"Downloading ZIP `graph_data_{search_space}.zip` from https://zenodo.org/ ...")
            response = requests.get(zip_url)
            response.raise_for_status()  # Raise an error if the download failed
        
            with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                print("Contents of the ZIP file:")
                for file_name in z.namelist():
                    print(f" - {file_name}")
        
                if target_file_name in z.namelist():
                    print(f"Extracting {target_file_name}...")
                    os.makedirs(target_directory, exist_ok=True)
                    z.extract(target_file_name, path=target_directory)
                    print(f"Saved '{target_file_name}' to '{target_directory}'.")
                else:
                    print(f"File '{target_file_name}' not found in the archive.")

In [9]:
target_directory = os.path.abspath('../../../data/processed_data/gcn_processed_data') 

## 3. Setup Configuration


We begin by choosing the specific search space in which we’ll reproduce the benchmark results.

In [10]:
# Define the search space
search_space = 'ls_a'

In [11]:
zenodo_load_data()

Contents of the ZIP file:
 - test_ls_a_squash.pt
 - train_ls_a_squash.pt
 - val_ls_a_squash.pt
Extracting test_ls_a_squash.pt...
Saved 'test_ls_a_squash.pt' to 'C:\SQuASH\data\processed_data\gcn_processed_data'.


Next, we define further configurations necessary for loading the model and dataset.

In [13]:
# also load the full JSON + timestamp if you like
config, gate_set_name, timestamp = prepare_paths_and_config(search_space, device)
print(f"\n=== Search‐space: {search_space} ({timestamp}) ===")
print("Config:")
print(json.dumps(config, indent=2, default=str))

set_seed(config["runseed"])


=== Search‐space: ls_a (2025-06-04_22-49-46) ===
Config:
{
  "device": "cpu",
  "seed": 42,
  "runseed": 42,
  "batch_size": 32,
  "num_workers": 0,
  "epochs": 100,
  "emb_dim": 1050,
  "layer_num": 8,
  "qubit_num": 4,
  "num_node_features": 8,
  "drop_ratio": 0.0644893118913786,
  "lr": 4.540520885756229e-05,
  "decay": 1.917208797826118e-06,
  "JK": "mean",
  "patience": 7,
  "metric": "spearman",
  "graph_pooling": "attention",
  "n_estimators": null,
  "max_depth": null,
  "random_state": null,
  "optuna_trials": null,
  "min_samples_split": null,
  "min_samples_leaf": null,
  "max_features": null,
  "n_jobs": null,
  "PATHS": {
    "optuna_studies": "C:\\Darya\\Projects\\EniQmA\\AK3\\Paper 2025 Squash\\SQuASH\\surrogate_models/tuning\\studies",
    "raw_data": "C:\\Darya\\Projects\\EniQmA\\AK3\\Paper 2025 Squash\\SQuASH\\data/raw_data/",
    "gcn_data": "C:\\Darya\\Projects\\EniQmA\\AK3\\Paper 2025 Squash\\SQuASH\\data/processed_data/gcn_processed_data",
    "rf_data": "C:\\Dar

## 4. Load Dataset and Benchmark GCN

In [16]:
dataset_names = [
    f"test_{search_space}_squash",
]
model_names = [
    f"gcn_{search_space}",
]

# Loop over each dataset/model pair
for data_set, model_name in zip(dataset_names, model_names):
    print(f"\n=== Dataset: {data_set} with Model: {model_name} ===")

    # load the data
    data_path = os.path.join(
        path_config.paths['gcn_data'],
        f"{data_set}.pt"
    )
    circuits   = torch.load(data_path, weights_only=False)          # list of Data objects

    loader     = DataLoader(circuits, batch_size=32, shuffle=False)

    # load model config
    cfg_path   = os.path.join(
        os.path.join(path_config.paths['benchmark_search_spaces'], f'{search_space}/surrogate_models/configs', f'{model_name}_config.json')
    )
    model_path = os.path.join(path_config.paths['benchmark_search_spaces'], f'{search_space}/surrogate_models', f"{model_name}.pth")
    model_cfg  = get_model_config_from_path(cfg_path, device)

    # instantiate RegGNN
    model = load_gnn_model(model_path, model_cfg)
    model.to(device).eval()

    # inference
    preds, labels  = evaluate_gnn(circuits, model, model_cfg)

    # compute and print metrics
    compute_metrics(preds, labels, label=model_name, tolerance=0.1)


=== Dataset: test_ls_a_squash with Model: gcn_ls_a ===
--- Metrics for gcn_ls_a ---
Samples: 17285
MSE:     0.0026
MAE:     0.0318
RMSE:    0.0510
R^2:     0.8927
Corr:    0.9470
Spearman: 0.9284
Accuracy (|err| <= 0.1): 94.10%

